In [1]:
# Cell 2
import re
import time
from datetime import datetime
import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

BASE_URL = "https://www.delhisldc.org/Loaddata.aspx"
START_DATE = "01-01-2022"   # dd-mm-yyyy
END_DATE   = "01-04-2026"   # dd-mm-yyyy
OUT_FILE   = "delhi_sldc_5min_2022_2026.csv"

# Website me common order yahi milta hai:
# Time, DELHI, BRPL, BYPL, NDPL, NDMC, MES
REGION_ORDER_ON_SITE = ["DELHI", "BRPL", "BYPL", "NDPL", "NDMC", "MES"]

retry = Retry(total=5, backoff_factor=1, status_forcelist=[429,500,502,503,504], allowed_methods=["GET"])
adapter = HTTPAdapter(max_retries=retry)

session = requests.Session()
session.mount("https://", adapter)
session.mount("http://", adapter)
session.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
})


In [3]:
# Cell 3
def clean_num(x):
    x = str(x).strip().replace(",", "")
    m = re.search(r"-?\d+(\.\d+)?", x)
    return float(m.group()) if m else None

def extract_timeslot(x):
    x = str(x).strip()
    # "00:00" ya "00:00-00:05" dono handle
    m = re.search(r"\b([0-2]?\d:[0-5]\d)\b", x)
    return m.group(1) if m else None

def parse_day_html(html, date_ddmmyyyy):
    soup = BeautifulSoup(html, "lxml")
    rows_out = []

    for table in soup.find_all("table"):
        for tr in table.find_all("tr"):
            cells = tr.find_all(["td", "th"])
            vals = [c.get_text(" ", strip=True) for c in cells]
            if len(vals) < 7:
                continue

            ts = extract_timeslot(vals[0])
            if not ts:
                continue

            nums = [clean_num(v) for v in vals[1:7]]
            if sum(v is not None for v in nums) < 4:  # weak row reject
                continue

            row = {
                "Date": date_ddmmyyyy,
                "TimeSlot": ts,
                "DELHI": nums[0],
                "BRPL": nums[1],
                "BYPL": nums[2],
                "NDPL": nums[3],
                "NDMC": nums[4],
                "MES": nums[5],
            }
            rows_out.append(row)

    if not rows_out:
        return pd.DataFrame(columns=["Date","TimeSlot","DELHI","BRPL","BYPL","NDPL","NDMC","MES"])

    df = pd.DataFrame(rows_out).drop_duplicates(subset=["Date", "TimeSlot"], keep="first")
    df["sort_key"] = pd.to_datetime(df["Date"] + " " + df["TimeSlot"], format="%d/%m/%Y %H:%M", errors="coerce")
    df = df.sort_values("sort_key").drop(columns=["sort_key"]).reset_index(drop=True)
    return df


In [5]:
# Cell 4
def fetch_one_day(d):
    mode = d.strftime("%d/%m/%Y")          # URL format
    date_out = d.strftime("%d/%m/%Y")      # output format (same as your sample)
    url = f"{BASE_URL}?mode={mode}"

    try:
        r = session.get(url, timeout=40)
        r.raise_for_status()
        df = parse_day_html(r.text, date_out)
        return df
    except Exception as e:
        print(f"[WARN] {mode} failed: {e}")
        return pd.DataFrame(columns=["Date","TimeSlot","DELHI","BRPL","BYPL","NDPL","NDMC","MES"])

start = datetime.strptime(START_DATE, "%d-%m-%Y")
end   = datetime.strptime(END_DATE, "%d-%m-%Y")
days = pd.date_range(start=start, end=end, freq="D")

all_parts = []
for d in tqdm(days, desc="Fetching"):
    part = fetch_one_day(d)
    if not part.empty:
        all_parts.append(part)
    time.sleep(0.2)  # polite delay

if not all_parts:
    raise RuntimeError("No rows extracted. Check connectivity / site blocking.")

final_df = pd.concat(all_parts, ignore_index=True)

# final order exactly sample style
final_df = final_df[["Date","TimeSlot","DELHI","BRPL","BYPL","NDPL","NDMC","MES"]]
final_df.to_csv(OUT_FILE, index=False)

print("Saved:", OUT_FILE)
print("Rows:", len(final_df))
final_df.head()


Fetching:   0%|          | 0/1552 [00:00<?, ?it/s]

Saved: delhi_sldc_5min_2022_2026.csv
Rows: 430782


,Date,TimeSlot,DELHI,BRPL,BYPL,NDPL,NDMC,MES
0,01/01/2022,00:00,2144.41,860.87,428.55,691.40,117.82,25.93
1,01/01/2022,00:05,2088.77,840.75,415.73,671.34,116.24,25.67
2,01/01/2022,00:10,2056.30,832.28,402.49,657.11,116.39,25.41
3,01/01/2022,00:15,2045.75,825.33,403.52,657.09,114.96,25.22
4,01/01/2022,00:20,2019.06,814.48,398.78,659.41,102.21,24.99


In [7]:
# Cell 5 (quick null + 5-min gap check)
print("Null counts:")
print(final_df.isna().sum())

tmp = final_df.copy()
tmp["dt"] = pd.to_datetime(tmp["Date"] + " " + tmp["TimeSlot"], format="%d/%m/%Y %H:%M", errors="coerce")
tmp = tmp.sort_values("dt")
tmp["gap_min"] = tmp.groupby("Date")["dt"].diff().dt.total_seconds() / 60
print("Common gaps:", tmp["gap_min"].dropna().mode().tolist()[:5])


Null counts:
Date        0
TimeSlot    0
DELHI       0
BRPL        0
BYPL        0
NDPL        0
NDMC        0
MES         0
dtype: int64
Common gaps: [5.0]
